# HRT POC – Overview

A simple proof-of-concept demonstrating how
HRT (Hierarchical Resolution Training) can improve model accuracy relative to the total number of tokens seen during training.


Rather than exposing a model to data directly at full complexity, HRT structures the learning process by layering data exposure progressively – reducing complexity first, then building up. This enables denser knowledge representations that preserve more information and generalise better, without needing more data or increasing model size. The goal is to avoid the wasted capacity of large networks that spend resources learning redundant features, instead of building compact, efficient understanding.


In [ ]:
!pip install torch datasets spacy tqdm tiktoken
!python -m spacy download en_core_web_sm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
import spacy
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
def process_dataset(dataset):
    processed = []
    for item in tqdm(dataset):
        doc = nlp(item["text"])
        tokens = [t.text for t in doc]
        pos = [t.pos_ for t in doc]
        processed.append({"tokens": tokens, "pos": pos})
    return processed



In [ ]:
from datasets import load_dataset
nlp = spacy.load("en_core_web_sm")

dataset = load_dataset("roneneldan/TinyStories", split="train[:12500]")
dataset = dataset.train_test_split(test_size=0.2, seed =  7777)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

train_processed = process_dataset(train_dataset)
val_processed   = process_dataset(val_dataset)

In [ ]:
# Build vocab
all_tokens = set()

for item in (train_processed):
    all_tokens.update(item["tokens"])

for item in (val_processed):
    all_tokens.update(item["tokens"])

MASK_TOKEN = "<mask>"
PAD_TOKEN = "<pad>"

vocab = {tok: i+2 for i, tok in enumerate(all_tokens)}
vocab[PAD_TOKEN] = 0
vocab[MASK_TOKEN] = 1

inv_vocab = {i: t for t, i in vocab.items()}

def encode(tokens):
    return [vocab.get(t, 0) for t in tokens]

In [ ]:
def apply_curriculum(tokens, pos, allowed_pos, mode="mask"):
    new_tokens = []

    for t, p in zip(tokens, pos):
        if p in allowed_pos:
            new_tokens.append(t)
        else:
            if mode == "remove":
                continue
            elif mode == "mask":
                new_tokens.append(MASK_TOKEN)

    return new_tokens

In [ ]:
ALL_POS = [
    "ADJ",    # adjective
    "ADP",    # adposition (prepositions, postpositions)
    "ADV",    # adverb
    "AUX",    # auxiliary verb (is, have, do, etc.)
    "CCONJ",  # coordinating conjunction (and, but, or)
    "DET",    # determiner (a, an, the)
    "INTJ",   # interjection (oh, wow)
    "NOUN",   # noun (common noun)
    "NUM",    # numeral
    "PART",   # particle (to, not)
    "PRON",   # pronoun
    "PROPN",  # proper noun (names)
    "PUNCT",  # punctuation
    "SCONJ",  # subordinating conjunction (because, although)
    "SYM",    # symbol ($, %, etc.)
    "VERB",   # verb (main verbs)
    "X",      # other (unknown, foreign words, etc.)
    "SPACE"   # space tokens
]

CONTENT_POS = [
    "NOUN",   # objects, concepts
    "PROPN",  # names, entities
    "VERB",   # actions
    "ADJ",    # properties
    "ADV",    # modifiers (often meaningful: "quickly", "deeply")
    "NUM",    # quantities
    "SYM",    # symbols (sometimes meaningful depending on task)
    "PUNCT",  # punctuation
    "X"       # unknown/foreign (optional, keep or drop based on use-case)
]

FILLER_POS = [
    "DET",    # the, a, an
    "ADP",    # in, on, at
    "AUX",    # is, have, do
    "CCONJ",  # and, but
    "SCONJ",  # because, although
    "PRON",   # he, she, it (context-dependent meaning)
    "PART",   # to, not
    "INTJ",   # oh, wow (optional, depends on task)

    "SPACE"   # spaces
]

In [ ]:
class CurriculumDataset:
    def __init__(self, data, mode="mask"):
        self.data = data
        self.mode = mode
        self.epoch = 0

        self.curriculum = [
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM", "PUNCT"},
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM", "PUNCT"},
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM", "PUNCT"},
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM", "PUNCT"},
          "FULL"
          ]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # 🔥 STANDARD MODE (baseline)
        if self.mode == "standard":
            tokens = item["tokens"]

        else:
            stage = self.curriculum[min(self.epoch, len(self.curriculum)-1)]

            if stage == "FULL":
                tokens = item["tokens"]
            else:
                tokens = apply_curriculum(
                    item["tokens"],
                    item["pos"],
                    stage,
                    self.mode
                )

        ids = encode(tokens)

        return torch.tensor(ids, dtype=torch.long)

In [ ]:
class CurriculumDataset1:
    def __init__(self, data, mode="mask"):
        self.data = data
        self.mode = mode
        self.epoch = 0

        self.curriculum = [
          {"VERB", "ADJ", "ADV", "PUNCT"},
          {"VERB", "ADJ", "ADV", "PUNCT"},
          {"VERB", "ADJ", "ADV", "PUNCT"},
          {"VERB", "ADJ", "ADV", "PUNCT"},
          "FULL"
          ]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # 🔥 STANDARD MODE (baseline)
        if self.mode == "standard":
            tokens = item["tokens"]

        else:
            stage = self.curriculum[min(self.epoch, len(self.curriculum)-1)]

            if stage == "FULL":
                tokens = item["tokens"]
            else:
                tokens = apply_curriculum(
                    item["tokens"],
                    item["pos"],
                    stage,
                    self.mode
                )

        ids = encode(tokens)

        return torch.tensor(ids, dtype=torch.long)

In [ ]:
class CurriculumDataset2:
    def __init__(self, data, mode="mask"):
        self.data = data
        self.mode = mode
        self.epoch = 0

        self.curriculum = [
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM"},
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM"},
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM"},
          {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "NUM", "SYM"},
          "FULL"
          ]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # 🔥 STANDARD MODE (baseline)
        if self.mode == "standard":
            tokens = item["tokens"]

        else:
            stage = self.curriculum[min(self.epoch, len(self.curriculum)-1)]

            if stage == "FULL":
                tokens = item["tokens"]
            else:
                tokens = apply_curriculum(
                    item["tokens"],
                    item["pos"],
                    stage,
                    self.mode
                )

        ids = encode(tokens)

        return torch.tensor(ids, dtype=torch.long)

In [ ]:
class CurriculumDataset3:
    def __init__(self, data, mode="mask"):
        self.data = data
        self.mode = mode
        self.epoch = 0

        self.curriculum = [
          {"VERB", "ADJ", "ADV"},
          {"VERB", "ADJ", "ADV"},
          {"VERB",  "ADJ", "ADV"},
          {"VERB",  "ADJ", "ADV"},
          "FULL"
          ]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # 🔥 STANDARD MODE (baseline)
        if self.mode == "standard":
            tokens = item["tokens"]

        else:
            stage = self.curriculum[min(self.epoch, len(self.curriculum)-1)]

            if stage == "FULL":
                tokens = item["tokens"]
            else:
                tokens = apply_curriculum(
                    item["tokens"],
                    item["pos"],
                    stage,
                    self.mode
                )

        ids = encode(tokens)

        return torch.tensor(ids, dtype=torch.long)

In [ ]:
def collate_fn(batch):
    batch = [b for b in batch if len(b) > 1]  # avoid empty
    return torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0)

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_emb=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_emb)
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, x):
        emb = self.embed(x)
        logits = self.lm_head(emb)
        return logits

In [ ]:
EPOCHS = 60
EDGE_EPOCHS = 4
# Define a subset percentage for data_to_use
subset_percentage_for_training = 1.0 # Adjust this percentage as needed (e.g., 0.5 for 50%)
subset_size_for_training = int(len(train_processed) * subset_percentage_for_training)
data_to_use = train_processed[:subset_size_for_training] # Using a subset of the processed data

print(f"Using {len(data_to_use)} items for training (approx. {subset_percentage_for_training*100}% of total processed data).")

In [ ]:
def freeze_output_layer():
    print("Freezing output layer...")
    for p in model.lm_head.parameters():
        p.requires_grad = False

In [ ]:
def unfreeze_output_layer():
    print("unfreeze")
    for p in model.lm_head.parameters():
        p.requires_grad = True

In [ ]:
def train_model(model, optimizer, train_loader, dataset_obj, val_dataset_obj, epochs, device, checkpoint_filename, freeze_flag=False):
    if freeze_flag:
        freeze_output_layer()

    val_loss_history = []
    tokens_seen_history = []
    cumulative_tokens_seen = 0

    for epoch in range(epochs):
        dataset_obj.set_epoch(epoch)
        total_train_loss = 0
        train_steps = 0

        if freeze_flag and epoch == EDGE_EPOCHS:
            unfreeze_output_layer()

        # ===== TRAINING LOOP =====
        model.train()
        for batch in train_loader:
            batch = batch.to(device)

            if batch.size(1) < 2:
                continue

            # Calculate tokens seen in this batch (excluding padding)
            # The target for prediction is batch[:, 1:]
            actual_tokens_in_batch = (batch[:, 1:] != 0).sum().item() # Count non-zero elements
            cumulative_tokens_seen += actual_tokens_in_batch

            logits = model(batch[:, :-1])

            loss = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                batch[:, 1:].reshape(-1),
                ignore_index=0
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            train_steps += 1

        avg_train_loss = total_train_loss / max(train_steps, 1)

        # ===== VALIDATION LOOP =====
        model.eval()
        total_val_loss = 0
        val_steps = 0

        with torch.no_grad():
            for item in val_dataset_obj:
                # Convert item to tensor batch
                batch = torch.tensor(item, dtype=torch.long).unsqueeze(0).to(device)
                if batch.size(1) < 2:
                    continue

                logits = model(batch[:, :-1])

                loss = nn.functional.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    batch[:, 1:].reshape(-1),
                    ignore_index=0
                )

                total_val_loss += loss.item()
                val_steps += 1

        avg_val_loss = total_val_loss / max(val_steps, 1)

        val_loss_history.append(avg_val_loss)
        tokens_seen_history.append(cumulative_tokens_seen)

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Cumulative Tokens: {cumulative_tokens_seen}")

    # Save the model and optimizer states after the last epoch
    torch.save({
        'epoch': epochs - 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss
        }, checkpoint_filename)

    print(f"Model and optimizer states saved to {checkpoint_filename}")
    return tokens_seen_history, val_loss_history


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_validation_loss(standard_tokens_seen, standard_val_loss, exp_tokens, exp_loss, label):

    # ===== FULL CURVE =====
    plt.figure(figsize=(10, 6))

    sns.lineplot(x=standard_tokens_seen, y=standard_val_loss, label="Standard Training")
    sns.lineplot(x=exp_tokens, y=exp_loss, label=label)

    plt.title(f"Validation Loss vs. Tokens Seen: Standard vs. {label}")
    plt.xlabel("Cumulative Tokens Seen")
    plt.ylabel("Validation Loss")
    plt.grid(True)
    plt.legend()
    plt.xscale('log')
    plt.tight_layout()
    plt.show()


    # ===== ZOOMED-IN LAST 20 EPOCHS =====
    zoom_n = 30

    std_tokens_zoom = standard_tokens_seen[-zoom_n:]
    std_loss_zoom = standard_val_loss[-zoom_n:]

    exp_tokens_zoom = exp_tokens[-zoom_n:]
    exp_loss_zoom = exp_loss[-zoom_n:]

    plt.figure(figsize=(10, 6))

    sns.lineplot(x=std_tokens_zoom, y=std_loss_zoom, label="Standard Training")
    sns.lineplot(x=exp_tokens_zoom, y=exp_loss_zoom, label=label)

    plt.title(f"Zoomed (Last {zoom_n} Epochs): Validation Loss vs. Tokens Seen")
    plt.xlabel("Cumulative Tokens Seen")
    plt.ylabel("Validation Loss")
    plt.grid(True)
    plt.legend()
    plt.xscale('log')  # keep consistent scale

    plt.tight_layout()
    plt.show()

In [ ]:
dataset_obj = CurriculumDataset(data_to_use, mode="standard")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

standard_tokens_seen, standard_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_standard.pt',
            freeze_flag= False)

In [ ]:
dataset_obj = CurriculumDataset(data_to_use, mode="remove")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

remove_tokens_seen, remove_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_remove.pt',
            freeze_flag= True)



In [ ]:
plot_validation_loss(standard_tokens_seen, standard_val_loss, remove_tokens_seen, remove_val_loss, "Remove (Curriculum 0)")

In [ ]:
dataset_obj = CurriculumDataset(data_to_use, mode="mask")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

mask_tokens_seen, mask_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_mask.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, mask_tokens_seen, mask_val_loss, "Mask (Curriculum 0)")

Circulam 1


In [ ]:
dataset_obj = CurriculumDataset1(data_to_use, mode="remove")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

remove1_tokens_seen, remove1_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_remove1.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, remove1_tokens_seen, remove1_val_loss, "Remove (Curriculum 1)")

In [ ]:
dataset_obj = CurriculumDataset1(data_to_use, mode="mask")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

mask1_tokens_seen, mask1_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_mask1.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, mask1_tokens_seen, mask1_val_loss, "Mask (Curriculum 1)")

ciriculam 2


In [ ]:
dataset_obj = CurriculumDataset2(data_to_use, mode="remove")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

remove2_tokens_seen, remove2_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_remove2.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, remove2_tokens_seen, remove2_val_loss, "Remove (Curriculum 2)")

In [ ]:
dataset_obj = CurriculumDataset2(data_to_use, mode="mask")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

mask2_tokens_seen, mask2_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_mask2.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, mask2_tokens_seen, mask2_val_loss, "Mask (Curriculum 2)")

ciriculam 3


In [ ]:
dataset_obj = CurriculumDataset3(data_to_use, mode="remove")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

remove3_tokens_seen, remove3_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_remove3.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, remove3_tokens_seen, remove3_val_loss, "Remove (Curriculum 3)")

In [ ]:
dataset_obj = CurriculumDataset3(data_to_use, mode="mask")  # change to "remove" to test
val_dataset   = CurriculumDataset(val_processed, mode="standard")  # always standard for validation

loader = DataLoader(dataset_obj, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = TinyGPT(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

mask3_tokens_seen, mask3_val_loss = train_model(model,
            optimizer,
            loader,
            dataset_obj,
            val_dataset,
            EPOCHS,
            device,
            'model_checkpoint_mask3.pt',
            freeze_flag= True)

plot_validation_loss(standard_tokens_seen, standard_val_loss, mask3_tokens_seen, mask3_val_loss, "Mask (Curriculum 3)")

In [ ]:
# Prepare data for plotting
plotting_data = [
    ("Remove (Curriculum 0)", remove_tokens_seen, remove_val_loss),
    ("Mask (Curriculum 0)", mask_tokens_seen, mask_val_loss),
    ("Remove (Curriculum 1)", remove1_tokens_seen, remove1_val_loss),
    ("Mask (Curriculum 1)", mask1_tokens_seen, mask1_val_loss),
    ("Remove (Curriculum 2)", remove2_tokens_seen, remove2_val_loss),
    ("Mask (Curriculum 2)", mask2_tokens_seen, mask2_val_loss),
    ("Remove (Curriculum 3)", remove3_tokens_seen, remove3_val_loss),
    ("Mask (Curriculum 3)", mask3_tokens_seen, mask3_val_loss)
]

for label, exp_tokens, exp_loss in plotting_data:
    plot_validation_loss(standard_tokens_seen, standard_val_loss, exp_tokens, exp_loss, label)